In [0]:
%pip install openpyxl


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("domain_name", "")
dbutils.widgets.text("sub_domain_name", "")

table_name = dbutils.widgets.get("table_name").strip()
domain_name = dbutils.widgets.get("domain_name").strip()
sub_domain_name = dbutils.widgets.get("sub_domain_name").strip()

# Used only for case-insensitive metadata lookup
table_name_lower = table_name.lower()
domain_name_lower = domain_name.lower()
sub_domain_name_lower = sub_domain_name.lower()

In [0]:
# Future alert implementation
#
# def trigger_alert(event_type, execution_id=None, asset_id=None, message=None):
#     """
#     Future implementation could:
#     - send email
#     - create UI notification
#     - trigger webhook
#     - notify the requesting user
#     """
#     pass

In [0]:
from pyspark.sql import functions as F

CATALOG = "mastercard_card_241614"
SCHEMA = "default"

assets_df = spark.table(
    f"{CATALOG}.{SCHEMA}.assets_m"
)

domains_df = spark.table(
    f"{CATALOG}.{SCHEMA}.domains_m"
)

sub_domains_df = spark.table(
    f"{CATALOG}.{SCHEMA}.sub_domains_m"
)

In [0]:
asset_df = (
    assets_df
    .filter(
        F.lower(
            F.regexp_extract(
                F.col("s3_path"),
                r"([^/]+)$",
                1
            )
        ) == table_name_lower
    )
)

In [0]:
display(
    asset_df.select(
        "asset_id",
        "asset_name",
        "domain_id",
        "sub_domain_id",
        "s3_path"
    )
)

asset_id,asset_name,domain_id,sub_domain_id,s3_path
26,01_TXN_transaction_status,1,1,source/TXN_PROCESSING/TRANSACTION_LIFECYCLE_STATUS/01_txn_transaction_status


In [0]:
asset_count = asset_df.count()

if asset_count == 0:
    message = (
        f"No asset found for table='{table_name}', "
        f"domain='{domain_name}', "
        f"sub_domain='{sub_domain_name}'"
    )

    # trigger_alert(
    #     event_type="ASSET_NOT_FOUND",
    #     message=message
    # )

    raise Exception(message)

if asset_count > 1:
    message = (
        f"Multiple assets found for table='{table_name}'. "
        f"Expected exactly one asset."
    )

    # trigger_alert(
    #     event_type="MULTIPLE_ASSETS_FOUND",
    #     message=message
    # )

    raise Exception(message)

asset = asset_df.first()

asset_id = asset.asset_id
asset_domain_id = asset.domain_id
asset_sub_domain_id = asset.sub_domain_id

In [0]:
domain = (
    domains_df
    .filter(
        F.col("domain_id") == asset_domain_id
    )
    .first()
)

if domain is None:
    message = (
        f"Domain metadata missing for asset_id={asset_id}"
    )

    # trigger_alert(
    #     event_type="DOMAIN_METADATA_MISSING",
    #     asset_id=asset_id,
    #     message=message
    # )

    raise Exception(message)

if domain.domain_name.lower() != domain_name_lower:
    message = (
        f"Domain mismatch for asset_id={asset_id}. "
        f"Expected='{domain.domain_name}', "
        f"Received='{domain_name}'"
    )

    # trigger_alert(
    #     event_type="DOMAIN_MISMATCH",
    #     asset_id=asset_id,
    #     message=message
    # )

    raise Exception(message)

In [0]:
sub_domain = (
    sub_domains_df
    .filter(
        (F.col("sub_domain_id") == asset_sub_domain_id) &
        (F.col("domain_id") == asset_domain_id)
    )
    .first()
)

if sub_domain is None:
    message = (
        f"Sub-domain metadata or domain relationship "
        f"missing for asset_id={asset_id}"
    )

    # trigger_alert(
    #     event_type="SUB_DOMAIN_METADATA_MISSING",
    #     asset_id=asset_id,
    #     message=message
    # )

    raise Exception(message)

In [0]:
if sub_domain.sub_domain_name.lower() != sub_domain_name_lower:
    message = (
        f"Sub-domain mismatch for asset_id={asset_id}. "
        f"Expected='{sub_domain.sub_domain_name}', "
        f"Received='{sub_domain_name}'"
    )

    # trigger_alert(
    #     event_type="SUB_DOMAIN_MISMATCH",
    #     asset_id=asset_id,
    #     message=message
    # )

    raise Exception(message)

In [0]:
print("✅ Asset validation passed")
print(f"   Asset ID     : {asset_id}")
print(f"   Domain       : {domain.domain_name}")
print(f"   Sub-domain   : {sub_domain.sub_domain_name}")
print(f"   Table        : {table_name}")

✅ Asset validation passed
   Asset ID     : 26
   Domain       : TXN_PROCESSING
   Sub-domain   : TRANSACTION_LIFECYCLE_STATUS
   Table        : 01_TXN_transaction_status


In [0]:
EXCEL_PATH="/Volumes/mastercard_card_241614/default/test_data/PCN_DQ_TestDataset (1).xlsx"

In [0]:
# ============================================================
# EXCEL -> SPARK DATAFRAME
# TYPE INFERENCE + METADATA-DRIVEN CASTING
# ============================================================

import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType


# ============================================================
# 1. READ EXCEL
# ============================================================

pdf = pd.read_excel(
    EXCEL_PATH,
    sheet_name=table_name
)


# ============================================================
# 2. DATE FORMAT DEFINITIONS
# ============================================================

DATE_FORMATS = [
    ("mixed", {"dayfirst": False}),
    ("mixed", {"dayfirst": True}),

    ("%Y-%m-%d %H:%M:%S", {}),
    ("%Y-%m-%d %H:%M", {}),
    ("%Y-%m-%d", {}),

    ("%Y/%m/%d %H:%M:%S", {}),
    ("%Y/%m/%d %H:%M", {}),
    ("%Y/%m/%d", {}),

    ("%d/%m/%Y %H:%M:%S", {}),
    ("%d/%m/%Y %H:%M", {}),
    ("%d/%m/%Y", {}),

    ("%d-%m-%Y %H:%M:%S", {}),
    ("%d-%m-%Y %H:%M", {}),
    ("%d-%m-%Y", {}),

    ("%m/%d/%Y %H:%M:%S", {}),
    ("%m/%d/%Y %H:%M", {}),
    ("%m/%d/%Y", {}),

    ("%m-%d-%Y %H:%M:%S", {}),
    ("%m-%d-%Y %H:%M", {}),
    ("%m-%d-%Y", {}),

    ("%Y%m%d", {}),
    ("%Y%m%d%H%M%S", {}),

    ("%Y-%m-%dT%H:%M:%S", {}),
    ("%Y-%m-%dT%H:%M:%SZ", {}),
    ("%Y-%m-%dT%H:%M:%S.%f", {}),
    ("%Y-%m-%dT%H:%M:%S.%fZ", {}),

    ("%d/%m/%Y %I:%M:%S %p", {}),
    ("%m/%d/%Y %I:%M:%S %p", {}),
    ("%d/%m/%Y %I:%M %p", {}),
    ("%m/%d/%Y %I:%M %p", {}),
]


# ============================================================
# 3. AUTO-INFER OBJECT COLUMNS
# ============================================================

for col in pdf.select_dtypes(include="object").columns:

    non_null = pdf[col].notna().sum()

    if non_null == 0:
        pdf[col] = pdf[col].astype(str)
        continue

    # Try numeric
    numeric = pd.to_numeric(
        pdf[col],
        errors="coerce"
    )

    if numeric.notna().sum() / non_null > 0.5:
        pdf[col] = numeric
        continue

    # Try datetime
    best_dt = None
    best_count = 0

    for fmt, kwargs in DATE_FORMATS:

        try:
            dt = pd.to_datetime(
                pdf[col],
                errors="coerce",
                format=fmt,
                **kwargs
            )

            count = dt.notna().sum()

            if count > best_count:
                best_count = count
                best_dt = dt

        except Exception:
            continue

    if (
        best_dt is not None
        and best_count / non_null > 0.5
    ):
        pdf[col] = best_dt
        continue

    # Keep as string
    pdf[col] = pdf[col].where(
        pdf[col].isna(),
        pdf[col].astype(str)
    )


# ============================================================
# 4. CREATE INITIAL SPARK DATAFRAME
# ============================================================

df = spark.createDataFrame(pdf)


# ============================================================
# 5. LOAD EXPECTED METADATA
# ============================================================

expected_metadata_df = (
    spark.table(
        f"{CATALOG}.{SCHEMA}.asset_columns_m"
    )
    .filter(
        F.col("asset_id") == asset_id
    )
)


# ============================================================
# 6. APPLY METADATA TYPES EXPLICITLY
# ============================================================

metadata_rows = (
    expected_metadata_df
    .select(
        "column_name",
        "data_type"
    )
    .collect()
)


for row in metadata_rows:

    metadata_column = row.column_name
    metadata_type = row.data_type.upper().strip()

    # Find actual Excel/Spark column
    actual_column = next(
        (
            c
            for c in df.columns
            if c.lower() == metadata_column.lower()
        ),
        None
    )

    if actual_column is None:
        continue


    # --------------------------------------------------------
    # STRING
    # --------------------------------------------------------

    if metadata_type == "STRING":

        df = df.withColumn(
            actual_column,
            F.col(actual_column).cast("string")
        )


    # --------------------------------------------------------
    # DECIMAL
    # --------------------------------------------------------

    elif metadata_type.startswith("DECIMAL"):

        # Extract precision and scale
        decimal_spec = metadata_type.replace(
            "DECIMAL(", ""
        ).replace(
            ")", ""
        )

        precision, scale = [
            int(x.strip())
            for x in decimal_spec.split(",")
        ]

        df = df.withColumn(
            actual_column,
            F.col(actual_column).cast(
                DecimalType(
                    precision,
                    scale
                )
            )
        )


    # --------------------------------------------------------
    # TIMESTAMP
    # --------------------------------------------------------

    elif metadata_type == "TIMESTAMP":

        df = df.withColumn(
            actual_column,
            F.col(actual_column).cast("timestamp")
        )


    # --------------------------------------------------------
    # DATE
    # --------------------------------------------------------

    elif metadata_type == "DATE":

        df = df.withColumn(
            actual_column,
            F.col(actual_column).cast("date")
        )


    # --------------------------------------------------------
    # OTHER TYPES
    # --------------------------------------------------------

    else:

        df = df.withColumn(
            actual_column,
            F.col(actual_column).cast(
                metadata_type.lower()
            )
        )


# ============================================================
# 7. SHOW FINAL SCHEMA
# ============================================================

print("========================================")
print("FINAL SPARK SCHEMA")
print("========================================")

df.printSchema()

EXCEL COLUMN NAMES
0 'STATUS_ID'
1 'TRANSACTION_IDSTATUS_RECORD_ID\tCARD_ID\tSTATUS\tPREVIOUS_STATUS\tCHANGED_AT\tCHANGED_BY\tCHANGE_REASON\tVALID_TRANSITION\tDQ_FLAG\nCST09850450\tCRD5810836075\tINACTIVE\tINACTIVE\t2024-11-26 09:26:17\tISSUER\tBLOCK\tY\tPASS\nCST00185590\tCRD3079159134\tPENDING\tPENDING\t2023-09-11 12:07:21\tCARDHOLDER\tEXPIRY\tY\tPASS\nCST17282584\tCRD6045470332\tACTIVE\tPENDING\t2024-07-29 21:19:06\tCARDHOLDER\tBLOCK\tY\tPASS\nCST16513718\tCRD4309579221\tCLOSED\tACTIVE\t2024-08-19 02:16:24\tAGENT\tRENEWAL\tY\tPASS\nCST29721366\tCRD0135552769\tCLOSED\tPENDING\t2024-03-12 22:53:26\tISSUER\tFRAUD\tY\tPASS\nCST39988757\tCRD3687347366\tACTIVE\tACTIVE\t2023-04-01 21:51:08\tSYSTEM\tUNBLOCK\tY\tPASS\nCST67032896\tCRD1912970399\tACTIVE\tCLOSED\t2023-04-23 23:58:40\tCARDHOLDER\tUNBLOCK\tN\tFAIL_INVALID_STATUS_TRANSITION\nCST24303891\tCRD8558658673\tPENDING\tSUSPENDED\t2023-07-30 08:28:30\tISSUER\tUNBLOCK\tY\tPASS\nCST33122429\tCRD7650118212\tSUSPENDED\tINACTIVE\t2023-12-22 09

In [0]:
# # ============================================================
# # VERIFY THE TWO PREVIOUSLY FAILING COLUMNS
# # ============================================================

# df.select(
#     "AMOUNT",
#     "PAN_LAST4",
#     "INITIATED_AT",
#     "AUTHORIZED_AT",
#     "CLEARED_AT",
#     "SETTLED_AT"
# ).printSchema()

In [0]:
expected_df = (
    spark.table(
        f"{CATALOG}.{SCHEMA}.asset_columns_m"
    )
    .filter(
        F.col("asset_id") == asset_id
    )
    .orderBy("ordinal_position")
)

In [0]:
expected_columns = [
    row.column_name
    for row in expected_df.select("column_name").collect()
]

actual_columns = df.columns

In [0]:
expected_set = {c.lower() for c in expected_columns}
actual_set = {c.lower() for c in actual_columns}

missing_columns = sorted(
    expected_set - actual_set
)

unexpected_columns = sorted(
    actual_set - expected_set
)

In [0]:
print(expected_columns)
print(actual_columns)

['STATUS_ID', 'TRANSACTION_ID', 'STATUS_CODE', 'STATUS_DESCRIPTION', 'STATUS_TIMESTAMP', 'RESPONSE_SOURCE', 'STATUS_CATEGORY', 'RETRY_COUNT', 'FINAL_STATUS', 'DQ_FLAG']
['STATUS_ID', 'TRANSACTION_IDSTATUS_RECORD_ID\tCARD_ID\tSTATUS\tPREVIOUS_STATUS\tCHANGED_AT\tCHANGED_BY\tCHANGE_REASON\tVALID_TRANSITION\tDQ_FLAG\nCST09850450\tCRD5810836075\tINACTIVE\tINACTIVE\t2024-11-26 09:26:17\tISSUER\tBLOCK\tY\tPASS\nCST00185590\tCRD3079159134\tPENDING\tPENDING\t2023-09-11 12:07:21\tCARDHOLDER\tEXPIRY\tY\tPASS\nCST17282584\tCRD6045470332\tACTIVE\tPENDING\t2024-07-29 21:19:06\tCARDHOLDER\tBLOCK\tY\tPASS\nCST16513718\tCRD4309579221\tCLOSED\tACTIVE\t2024-08-19 02:16:24\tAGENT\tRENEWAL\tY\tPASS\nCST29721366\tCRD0135552769\tCLOSED\tPENDING\t2024-03-12 22:53:26\tISSUER\tFRAUD\tY\tPASS\nCST39988757\tCRD3687347366\tACTIVE\tACTIVE\t2023-04-01 21:51:08\tSYSTEM\tUNBLOCK\tY\tPASS\nCST67032896\tCRD1912970399\tACTIVE\tCLOSED\t2023-04-23 23:58:40\tCARDHOLDER\tUNBLOCK\tN\tFAIL_INVALID_STATUS_TRANSITION\nCST243038

In [0]:
# ============================================================
# 10. DATA TYPE VALIDATION
# ============================================================

expected_schema = {
    row.column_name.lower(): row.data_type
    for row in expected_df.select(
        "column_name",
        "data_type"
    ).collect()
}

actual_schema = {
    field.name.lower(): field.dataType.simpleString()
    for field in df.schema.fields
}


def normalize_data_type(data_type):
    data_type = data_type.upper().strip()

    mappings = {
        "INTEGER": "INT",
        "LONG": "BIGINT",
        "BIGINT": "BIGINT",
        "DOUBLE": "DOUBLE",
        "FLOAT": "FLOAT",
        "BOOLEAN": "BOOLEAN",
        "BOOL": "BOOLEAN",
        "STRING": "STRING",
        "DATE": "DATE",
        "TIMESTAMP": "TIMESTAMP"
    }

    return mappings.get(data_type, data_type)


data_type_mismatches = []

for column_name in expected_schema:

    # Only compare columns that actually exist
    # Column existence was already validated above.
    if column_name in actual_schema:

        expected_type = normalize_data_type(
            expected_schema[column_name]
        )

        actual_type = normalize_data_type(
            actual_schema[column_name]
        )

        if expected_type != actual_type:

            data_type_mismatches.append({
                "column_name": column_name,
                "expected_data_type": expected_schema[column_name],
                "actual_data_type": actual_schema[column_name]
            })


if data_type_mismatches:

    message = (
        f"Data type validation failed for asset_id={asset_id}. "
        f"Mismatches={data_type_mismatches}"
    )

    # Future alert
    # trigger_alert(
    #     event_type="DATA_TYPE_VALIDATION_FAILED",
    #     asset_id=asset_id,
    #     message=message
    # )

    raise Exception(message)


print("✅ Data type validation passed")

✅ Data type validation passed


In [0]:
# ============================================================
# NULLABILITY / REQUIRED-FIELD VALIDATION
# ============================================================

nullability_mismatches = []

actual_columns = {
    c.lower(): c
    for c in df.columns
}

for row in expected_df.select(
    "column_name",
    "is_nullable"
).collect():

    column_name = row["column_name"]
    expected_nullable = row["is_nullable"]

    # Missing columns are handled by column-existence validation
    if column_name.lower() not in actual_columns:
        continue

    actual_column = actual_columns[column_name.lower()]

    # Only NOT NULL metadata requires a NULL-value check
    if expected_nullable is False:

        null_count = df.filter(
            F.col(actual_column).isNull()
        ).count()

        if null_count > 0:

            nullability_mismatches.append({
                "column_name": column_name,
                "expected_nullable": False,
                "null_count": null_count
            })


if nullability_mismatches:

    raise Exception(
        f"Nullability validation failed for asset_id={asset_id}. "
        f"Mismatches={nullability_mismatches}"
    )

print(
    f"✅ Nullability validation passed for asset_id={asset_id}"
)

✅ Nullability validation passed for asset_id=26


In [0]:
# ============================================================
# 12. ORDINAL POSITION VALIDATION
# ============================================================

expected_ordinal = {
    row.column_name.lower(): row.ordinal_position
    for row in expected_df.select(
        "column_name",
        "ordinal_position"
    ).collect()
}

actual_ordinal = {
    column_name.lower(): position
    for position, column_name in enumerate(
        df.columns,
        start=1
    )
}


ordinal_mismatches = []

for column_name in expected_ordinal:

    if column_name in actual_ordinal:

        expected_position = expected_ordinal[column_name]
        actual_position = actual_ordinal[column_name]

        if expected_position != actual_position:

            ordinal_mismatches.append({
                "column_name": column_name,
                "expected_position": expected_position,
                "actual_position": actual_position
            })


if ordinal_mismatches:

    message = (
        f"Ordinal position validation failed for asset_id={asset_id}. "
        f"Mismatches={ordinal_mismatches}"
    )

    # Future alert
    # trigger_alert(
    #     event_type="ORDINAL_POSITION_VALIDATION_FAILED",
    #     asset_id=asset_id,
    #     message=message
    # )

    raise Exception(message)


print("✅ Ordinal position validation passed")

✅ Ordinal position validation passed


In [0]:
if missing_columns or unexpected_columns:

    message = (
        f"Schema validation failed for asset_id={asset_id}. "
        f"Missing columns={missing_columns}, "
        f"Unexpected columns={unexpected_columns}"
    )

    # trigger_alert(
    #     event_type="SCHEMA_VALIDATION_FAILED",
    #     asset_id=asset_id,
    #     message=message
    # )

    raise Exception(message)

---------------------------------------------------------------------------
Exception                                 Traceback (most recent call last)
File <command-7304211177227014>, line 15
      3 message = (
      4     f"Schema validation failed for asset_id={asset_id}. "
      5     f"Missing columns={missing_columns}, "
      6     f"Unexpected columns={unexpected_columns}"
      7 )
      9 # trigger_alert(
     10 #     event_type="SCHEMA_VALIDATION_FAILED",
     11 #     asset_id=asset_id,
     12 #     message=message
     13 # )
---> 15 raise Exception(message)

Exception: Schema validation failed for asset_id=26. Missing columns=['transaction_id'], Unexpected columns=['transaction_idstatus_record_id\tcard_id\tstatus\tprevious_status\tchanged_at\tchanged_by\tchange_reason\tvalid_transition\tdq_flag\ncst09850450\tcrd5810836075\tinactive\tinactive\t2024-11-26 09:26:17\tissuer\tblock\ty\tpass\ncst00185590\tcrd3079159134\tpending\tpending\t2023-09-11 12:07:21\tcardholder\texpi

In [0]:
print("✅ Schema validation passed")
print(f"   Asset ID     : {asset_id}")
print(f"   Domain       : {domain_name}")
print(f"   Sub-domain   : {sub_domain_name}")
print(f"   Table        : {table_name}")
print(f"   Columns      : {len(actual_columns)}")

In [0]:
dbutils.notebook.exit("PASS")